# Gradient Boosting:

## The idea:

Gradient boosting builds a strong model by adding **small decision trees one at a time**, where each new tree is trained to fix the **errors the current ensemble still makes**. Trees are added sequentially, and the prediction is the **sum** of all the trees' outputs, each scaled by a small learning rate.

Compare this with AdaBoost, which fixes errors by re-weighting samples. Gradient boosting fixes them by **fitting the errors directly**.

## Why "gradient"?:

For squared error, the loss for one sample is ½ · (y − F)², where F is the model's current prediction. Its gradient with respect to F is −(y − F), so the **negative gradient is exactly the residual, y − F**.

So when each tree fits the residuals, the ensemble takes a **gradient descent step**. The difference from the gradient descent you already know is that it moves in **function space**:
- In ordinary gradient descent, you update parameters (weights) with `w ← w − η · gradient`.
- In gradient boosting, you update the prediction function with `F ← F + η · tree`. The tree approximates the negative gradient, and the learning rate η is the step size.

This view is what makes the method work for **any differentiable loss**. You just fit trees to the negative gradient of that loss, called the **pseudo-residuals**.

## The algorithm (regression, squared error):

1. **Start with a constant:** F₀(x) = mean of y.
2. For each round m = 1 … M:
   1. Compute the residuals: rᵢ = yᵢ − Fₘ₋₁(xᵢ).
   2. Fit a small tree hₘ to those residuals.
   3. Update: **Fₘ(x) = Fₘ₋₁(x) + η · hₘ(x)**.
3. The final model is F_M(x).

## A worked example:

Take 6 points with x = 1 to 6 and y = [10, 14, 16, 30, 34, 40]. Use stumps and a learning rate of η = 0.5. (A typical value is 0.1, but 0.5 keeps the numbers readable.)

**Start:** F₀ = mean(y) = **24** for every point. The mean squared error (MSE) is 125.3.

**Round 1:**
- Residuals: [−14, −10, −8, 6, 10, 16].
- The best stump splits at x ≤ 3.5. Its left leaf predicts the mean of the left residuals (−10.67), and its right leaf predicts +10.67.
- Update: the first three points become 24 + 0.5 × (−10.67) = **18.67**, and the last three become 24 + 0.5 × 10.67 = **29.33**. The MSE drops to 40.0.

**Round 2:**
- New residuals: y − F₁ = [−8.67, −4.67, −2.67, 0.67, 4.67, 10.67].
- This stump splits at x ≤ 4.5, with leaf values −3.83 (left) and +7.67 (right).
- Update: F₂ = [16.75, 16.75, 16.75, 27.42, 33.17, 33.17]. The MSE drops to 18.0.

| After round | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| MSE | 125.3 | 40.0 | 18.0 | 9.2 |

Each round chips away at what is left, and the learning rate stops any single tree from correcting too aggressively.

## Classification:

The same recipe works for classification with **log loss**:
- The model's raw output F is the **log-odds**, and the predicted probability is p = sigmoid(F).
- F₀ is the log-odds of the class proportions in the training data.
- The pseudo-residual is **y − p**: how far the predicted probability is from the true label (0 or 1).
- Each tree fits those pseudo-residuals, its leaf values are adjusted to best reduce the log loss within each leaf, and the tree is added scaled by η.
- For multi-class problems, one tree per class is built in each round.

## Choosing a loss:

| Task | Common losses |
|---|---|
| Regression | `squared_error` (default), `absolute_error` and `huber` (more robust to outliers), `quantile` (predict a percentile) |
| Classification | `log_loss` (default), `exponential` |

With `loss='exponential'`, gradient boosting recovers **AdaBoost**. So AdaBoost is a special case of gradient boosting.

## In scikit-learn:

```python
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
)
```

Those values are the defaults (I checked them on scikit-learn 1.8). `GradientBoostingRegressor` works the same way. For large datasets, `HistGradientBoostingClassifier` and `HistGradientBoostingRegressor` are much faster, because they bin the features into histograms first.

## Key hyperparameters:

| Parameter | Effect |
|---|---|
| `n_estimators` | Number of trees. More trees fit the data harder, so too many can overfit. |
| `learning_rate` | Shrinks every tree's contribution. Smaller values need more trees but usually generalise better. |
| `max_depth` | Depth of each tree (default 3). Shallow trees are the norm, since each tree should be a weak learner. |
| `subsample` | Fraction of rows each tree sees. Below 1.0 it is called **stochastic gradient boosting** and adds regularisation. |
| `min_samples_leaf`, `max_features` | More ways to limit tree complexity and add randomness. |
| `n_iter_no_change` | Early stopping: stop adding trees when a held-out validation score stops improving. |

A common recipe is to set a small learning rate (0.05 to 0.1), use early stopping to pick the number of trees, and tune `max_depth` and `subsample`. You can also use `staged_predict` to see how the score changes as trees are added.

## Overfitting:

Unlike a random forest, **adding more trees can overfit**, because each new tree keeps chasing the remaining error, including noise. The main defences are:
- a small learning rate
- shallow trees
- subsampling
- early stopping

## Comparison:

| | Random forest | AdaBoost | Gradient boosting |
|---|---|---|---|
| Training | parallel | sequential | sequential |
| Trees | deep, independent | stumps | small trees (depth ~3) |
| How errors are corrected | not at all, averaged out | re-weight samples | fit the residuals / gradients |
| Combining | equal vote | α-weighted vote | sum scaled by learning rate |
| Mainly reduces | variance | bias | bias |
| Loss | fixed | exponential | any differentiable loss |

## Strengths and weaknesses:

- **Good:** it often gives the best accuracy on tabular data, handles mixed feature types, doesn't need feature scaling, supports many loss functions, and gives feature importances.
- **Bad:** it needs careful tuning, it trains sequentially so it can be slow, and it overfits more easily than a random forest. Squared-error versions are also sensitive to outliers, so use `huber` or `absolute_error` if that matters.

---